In [ ]:
%pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
import numpy as np
import evaluate

class InputDataset(Dataset):
  def __init__(self, df, tokenizer, max_len=128):
    self.encodings = tokenizer(
      list(df["text"]),
      truncation=True,
      padding=True,
      max_length=max_len,
      return_tensors="pt"
    )
    self.labels = list(df["label"])

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return {
      "input_ids": self.encodings["input_ids"][idx],
      "attention_mask": self.encodings["attention_mask"][idx],
      "labels": torch.tensor(self.labels[idx], dtype=torch.long)
    }

# bert to rank the jokes/lyrics
def train_model(filename, labelname, output_dir, num_samples=None, learning_rate=5e-5, weight_decay=0.01):
  df = pd.read_csv(filename)
  if num_samples:
    df = df.sample(n=num_samples, random_state=42)
  df["label"] = df[labelname].astype(int)

  train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

  tokenizer = DistilBertTokenizer.from_pretrained("google/bert_uncased_L-2_H-128_A-2")
  model = DistilBertForSequenceClassification.from_pretrained("google/bert_uncased_L-2_H-128_A-2", num_labels=2)

  train_dataset = InputDataset(train_df, tokenizer)
  eval_dataset = InputDataset(eval_df,  tokenizer)

  accuracy_metric = evaluate.load("accuracy")
  f1_metric = evaluate.load("f1")

  def compute_metrics(eval_pred):
      logits, labels = eval_pred
      predictions = np.argmax(logits, axis=-1)
      accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
      f1 = f1_metric.compute(predictions=predictions, references=labels)
      return {"accuracy": accuracy["accuracy"], "f1": f1["f1"]}

  training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_dir="./logs",
    learning_rate=learning_rate,
    weight_decay=weight_decay,
  )

  trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
  )

  trainer.train()
  model.save_pretrained(output_dir)
  tokenizer.save_pretrained(output_dir)

In [ ]:
from transformers import pipeline

def classify_text(texts, model_dir):
  classifier = pipeline(
    "text-classification",
    model=model_dir,
    tokenizer=model_dir,
    return_all_scores=True
  )

  def score(text):
    output = classifier(text)
    scores = output[0]
    # if top label is LABEL_1, use its score. if LABEL_0 won, humor score is 1 - that score
    if scores["label"] == "LABEL_1":
      return scores["score"]
    else:
      return 1 - scores["score"]

  ranked = sorted(texts, key=score, reverse=True)
  for text in texts:
    print(f"{score(text):.3f}  {text}")

In [ ]:
import re

def normalize_lyric(text):
  text = text.lower()
  text = re.sub(r"[^\w\s]", "", text) # get rid of punctuation
  text = re.sub(r"\s+", " ", text).strip()
  return text

In [ ]:
train_model(
  filename="joke_dataset.csv",
  labelname="humor",
  output_dir="content/drive/MyDrive/CS6320Models/joke",
  num_samples=100000
)

You are using a model of type bert to instantiate a model of type distilbert. This is not supported for all configurations of models and can yield errors.


Loading weights: 0it [00:00, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                                          | Status     | 
-------------------------------------------------------------+------------+-
bert.encoder.layer.{0, 1}.attention.output.dense.bias        | UNEXPECTED | 
bert.encoder.layer.{0, 1}.output.LayerNorm.weight            | UNEXPECTED | 
bert.encoder.layer.{0, 1}.intermediate.dense.weight          | UNEXPECTED | 
bert.embeddings.LayerNorm.weight                             | UNEXPECTED | 
bert.encoder.layer.{0, 1}.attention.self.value.bias          | UNEXPECTED | 
bert.encoder.layer.{0, 1}.attention.self.value.weight        | UNEXPECTED | 
bert.encoder.layer.{0, 1}.attention.output.dense.weight      | UNEXPECTED | 
cls.predictions.transform.dense.weight                       | UNEXPECTED | 
bert.encoder.layer.{0, 1}.attention.output.LayerNorm.bias    | UNEXPECTED | 
cls.seq_relationship.weight                                  | UN

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.114692,0.126266,0.957850,0.958548
2,0.095919,0.109080,0.963750,0.963375
3,0.078810,0.117144,0.964700,0.964686
4,0.065331,0.118925,0.962550,0.962488
5,0.053715,0.143967,0.963250,0.963354


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
items = [
    # jokes
    "Why don't scientists trust atoms? Because they make up everything!",
    "I told my wife she was drawing her eyebrows too high. She looked surprised.",
    "Why did the scarecrow win an award? Because he was outstanding in his field!",
    "I'm reading a book about anti-gravity. It's impossible to put down.",
    "Why did the bicycle fall over? Because it was two-tired!",

    # nonjokes
    "The Eiffel Tower was completed in 1889 and stands 330 meters tall.",
    "Photosynthesis is the process by which plants convert sunlight into energy.",
    "The Amazon River discharges more water than any other river on Earth.",
    "Python was created by Guido van Rossum and first released in 1991.",
    "The human brain contains approximately 86 billion neurons.",
]
classify_text(items, "/content/drive/MyDrive/CS6320Models/joke")

Loading weights:   0%|          | 0/40 [00:00<?, ?it/s]

0.997  Why don't scientists trust atoms? Because they make up everything!
0.999  I told my wife she was drawing her eyebrows too high. She looked surprised.
0.998  Why did the scarecrow win an award? Because he was outstanding in his field!
0.987  I'm reading a book about anti-gravity. It's impossible to put down.
0.998  Why did the bicycle fall over? Because it was two-tired!
0.976  The Eiffel Tower was completed in 1889 and stands 330 meters tall.
0.912  Photosynthesis is the process by which plants convert sunlight into energy.
0.997  The Amazon River discharges more water than any other river on Earth.
0.989  Python was created by Guido van Rossum and first released in 1991.
0.104  The human brain contains approximately 86 billion neurons.


In [ ]:
import pandas as pd

df = pd.read_csv("lyric_dataset.csv")
df["text"] = df["text"].apply(normalize_lyric)
df.to_csv("lyrics_clean.csv", index=False)

train_model(
  filename="lyrics_clean.csv",
  labelname="label",
  output_dir="content/drive/MyDrive/CS6320Models/lyric",
  num_samples=25000
)

You are using a model of type bert to instantiate a model of type distilbert. This is not supported for all configurations of models and can yield errors.


Loading weights: 0it [00:00, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                                          | Status     | 
-------------------------------------------------------------+------------+-
bert.encoder.layer.{0, 1}.output.LayerNorm.bias              | UNEXPECTED | 
bert.embeddings.LayerNorm.bias                               | UNEXPECTED | 
cls.seq_relationship.bias                                    | UNEXPECTED | 
cls.predictions.transform.dense.bias                         | UNEXPECTED | 
bert.encoder.layer.{0, 1}.output.dense.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                                       | UNEXPECTED | 
bert.embeddings.LayerNorm.weight                             | UNEXPECTED | 
bert.encoder.layer.{0, 1}.attention.self.value.bias          | UNEXPECTED | 
bert.encoder.layer.{0, 1}.attention.self.key.weight          | UNEXPECTED | 
bert.encoder.layer.{0, 1}.attention.self.value.weight        | UN

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.516210,0.299260,0.878600,0.882751
2,0.278059,0.266256,0.892400,0.896697
3,0.207549,0.247652,0.902000,0.901725
4,0.148074,0.281814,0.902800,0.902016
5,0.130723,0.303283,0.905200,0.903186
6,0.115235,0.324662,0.903400,0.903957


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
items = [
    "Is this the real life? Is this just fantasy?",
    "We will, we will rock you!",
    "I keep on fallin' in and out of love with you",
    "Sweet Caroline, good times never seemed so good",
    "Don't stop believin', hold on to the feelin'",

    "The quick brown fox jumps over the lazy dog",
    "Please remember to take out the trash on Tuesday",
    "The meeting has been rescheduled to 3pm tomorrow",
    "Water boils at 100 degrees Celsius at sea level",
    "The library closes at 9pm on weekdays",
]
items = [normalize_lyric(item) for item in items]
classify_text(items, "/content/drive/MyDrive/CS6320Models/lyric")

Loading weights:   0%|          | 0/40 [00:00<?, ?it/s]

0.134  is this the real life is this just fantasy
0.989  we will we will rock you
0.989  i keep on fallin in and out of love with you
0.986  sweet caroline good times never seemed so good
0.988  dont stop believin hold on to the feelin
0.058  the quick brown fox jumps over the lazy dog
0.733  please remember to take out the trash on tuesday
0.014  the meeting has been rescheduled to 3pm tomorrow
0.011  water boils at 100 degrees celsius at sea level
0.013  the library closes at 9pm on weekdays
